# LangChain Messages

## Topics covered
- What is a Message?
- Text prompts vs Message prompts
- System, Human, AI, and Tool messages
- Message content and metadata
- Tool calls
- Token usage
- Streaming and chunks
- Multimodal messages
- Content blocks
- Serialization
- Using messages with chat models


## 1. What is a Message?

A **Message** is a basic piece of information sent to or received from an AI model.

A message mainly has:

- **Role** → who sent it
- **Content** → the actual information
- **Metadata** → extra information such as IDs or token usage

LangChain uses a standard message format so the same idea can work with different model providers.

In [ ]:
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, AIMessage, SystemMessage

# Example model
# model = init_chat_model("gpt-5-nano")

system_msg = SystemMessage("You are a helpful assistant.")
human_msg = HumanMessage("Hello, how are you?")

messages = [system_msg, human_msg]

# response = model.invoke(messages)
# print(response)

print("Messages are sent to a chat model as a list.")

## 2. Text Prompt

A text prompt is simply a string.

Use it when you have **one simple request** and do not need conversation history.

Example:

`"Write a haiku about spring"`

### Use text prompts when:
- You have one standalone question
- You do not need previous conversation
- You want very simple code

In [ ]:
# Simple text prompt
# response = model.invoke("Write a haiku about spring")
# print(response)

print("A string can be used as a simple user prompt.")

## 3. Message Prompt

A message prompt is a list of message objects.

It is useful when you need:
- Conversation history
- System instructions
- Images, audio, or files

Example roles:
- `system` → instructions for the model
- `user` → user message
- `assistant` → previous AI response

In [ ]:
messages = [
    SystemMessage("You are a poetry expert."),
    HumanMessage("Write a haiku about spring."),
    AIMessage("Cherry blossoms bloom...")
]

# response = model.invoke(messages)

print("Message prompts are useful for conversations and instructions.")

## 4. Dictionary Format

Messages can also be written as dictionaries.

The common format is:

- `role`
- `content`

This is similar to the OpenAI chat format.

In [ ]:
messages = [
    {"role": "system", "content": "You are a poetry expert."},
    {"role": "user", "content": "Write a haiku about spring."},
    {"role": "assistant", "content": "Cherry blossoms bloom..."}
]

# response = model.invoke(messages)

print("Dictionary format is another way to represent messages.")

## 5. Types of Messages

LangChain mainly uses four message types:

| Message | Simple meaning |
|---|---|
| `SystemMessage` | Gives instructions to the model |
| `HumanMessage` | Represents the user's input |
| `AIMessage` | Represents the model's answer |
| `ToolMessage` | Contains the result of a tool call |

## 6. SystemMessage

A `SystemMessage` tells the model **how it should behave**.

You can use it to:
- Set the model's role
- Set the tone
- Give rules
- Give instructions

For example, you can tell the model to act like a Python developer.

In [ ]:
system_msg = SystemMessage(
    "You are a helpful coding assistant."
)

messages = [
    system_msg,
    HumanMessage("How do I create a REST API?")
]

# response = model.invoke(messages)

print("SystemMessage gives instructions to the model.")

## 7. HumanMessage

A `HumanMessage` represents what the **user says or sends**.

It can contain:
- Text
- Images
- Audio
- Files
- Other multimodal content

A simple string can also be used as a shortcut for one `HumanMessage`.

In [ ]:
# Message object
message = HumanMessage("What is machine learning?")

# String shortcut
# response = model.invoke("What is machine learning?")

print(message.content)

## 8. HumanMessage Metadata

A HumanMessage can also have optional information such as:

- `name` → identify a user
- `id` → unique message ID

The behavior of `name` can be different depending on the model provider.

In [ ]:
human_msg = HumanMessage(
    content="Hello!",
    name="alice",
    id="msg_123"
)

print(human_msg.name)
print(human_msg.id)

## 9. AIMessage

An `AIMessage` represents the **answer from the AI model**.

It can contain:
- Text
- Tool calls
- Metadata
- Token usage
- Multimodal content

When a chat model is invoked, the response is normally an `AIMessage`.

In [ ]:
# Example:
# response = model.invoke("Explain AI")
# print(type(response))

# You can also create an AI message yourself:
ai_msg = AIMessage("I'd be happy to help!")

print(ai_msg.content)

## 10. Important AIMessage Fields

Some useful fields are:

- `text` → text content
- `content` → raw content
- `content_blocks` → standardized content
- `tool_calls` → tools requested by the model
- `id` → message ID
- `usage_metadata` → token usage
- `response_metadata` → provider response information

## 11. Tool Calls

Sometimes the AI needs a tool to complete a task.

For example:

**User:** What's the weather in Paris?

**AI:** I need to use the weather tool.

The tool call can contain:
- Tool name
- Arguments
- Tool call ID

These tool calls are stored inside the `AIMessage`.

In [ ]:
def get_weather(location: str) -> str:
    """Get the weather at a location."""
    return f"Weather information for {location}"

# model_with_tools = model.bind_tools([get_weather])
# response = model_with_tools.invoke("What's the weather in Paris?")

# for tool_call in response.tool_calls:
#     print(tool_call["name"])
#     print(tool_call["args"])
#     print(tool_call["id"])

print("AIMessage can contain tool calls.")

## 12. ToolMessage

A `ToolMessage` contains the **result returned by a tool**.

Simple flow:

1. User asks a question
2. AI asks for a tool
3. Tool runs
4. Tool returns a result in `ToolMessage`
5. AI uses that result to answer the user

The `tool_call_id` must match the ID of the tool call.

In [ ]:
from langchain.messages import ToolMessage

ai_message = AIMessage(
    content=[],
    tool_calls=[{
        "name": "get_weather",
        "args": {"location": "San Francisco"},
        "id": "call_123"
    }]
)

tool_message = ToolMessage(
    content="Sunny, 72°F",
    tool_call_id="call_123"
)

print(tool_message.content)

## 13. ToolMessage Artifact

A ToolMessage can have an `artifact`.

An artifact stores extra information that is **not sent to the model** but can still be used by your application.

For example, a search tool might return:
- Text for the model
- Document ID for your application
- Page number for displaying the source

In [ ]:
tool_message = ToolMessage(
    content="Useful text from a document",
    tool_call_id="call_123",
    name="search_books",
    artifact={
        "document_id": "doc_123",
        "page": 0
    }
)

print(tool_message.artifact)

## 14. Message Content

The `content` of a message is the actual information sent to the model.

It can be:

1. A simple string
2. A provider-specific list of content
3. A list of LangChain standard content blocks

This makes it possible to work with text and other data types.

In [ ]:
# Simple text
human_message = HumanMessage("Hello!")

print(human_message.content)

## 15. Multimodal Messages

**Multimodal** means working with different types of data.

Examples:
- Text
- Image
- Audio
- Video
- Files such as PDFs

Not every model supports every type, so you should check the model provider's supported formats.

In [ ]:
# Example: image content
message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Describe this image."},
        {
            "type": "image",
            "url": "https://example.com/image.jpg"
        }
    ]
}

print("A message can contain text and image content.")

## 16. File, Audio, and Video Content

LangChain can represent files, audio, and video using content blocks.

A file can be provided using things such as:
- URL
- Base64 data
- Provider-managed file ID

The exact requirements depend on the model provider.

**Important:** Not every model supports every file type.

## 17. Content Blocks

Content blocks give a **standard format** for different types of message content.

Examples include:

- `text` → normal text
- `reasoning` → reasoning information
- `image` → image data
- `audio` → audio data
- `video` → video data
- `file` → files
- `tool_call` → tool calls
- `server_tool_call` → server-side tool calls

This helps make content more consistent across different providers.

In [ ]:
# Example of a text content block
text_block = {
    "type": "text",
    "text": "Hello world"
}

# Example of an image content block
image_block = {
    "type": "image",
    "url": "https://example.com/image.jpg"
}

print(text_block)
print(image_block)

## 18. Reasoning Content Block

A `reasoning` content block represents reasoning information provided by a model.

Its type is:

`"reasoning"`

It can contain a `reasoning` field and additional provider-specific information.

The exact reasoning information depends on the model/provider.

## 19. Tool Call Content Blocks

A `tool_call` block represents a function/tool call.

It contains:

- `type` → `"tool_call"`
- `name` → tool name
- `args` → arguments for the tool
- `id` → unique call ID

During streaming, a partial tool call can appear as a `tool_call_chunk`.

In [ ]:
tool_call_block = {
    "type": "tool_call",
    "name": "search",
    "args": {"query": "weather"},
    "id": "call_123"
}

print(tool_call_block)

## 20. Server-Side Tool Calls

Some providers can run tools on their own servers.

There are special content blocks for this:

- `server_tool_call`
- `server_tool_call_chunk`
- `server_tool_result`

A `server_tool_result` can show whether the tool execution was successful or failed.

## 21. Provider-Specific Blocks

LangChain also allows provider-specific content.

The standard type is:

`"non_standard"`

This can be useful when a provider has a feature that does not fit the common LangChain format.

## 22. Streaming and AIMessageChunk

When a model streams its response, you do not receive one complete `AIMessage` immediately.

Instead, you receive smaller **`AIMessageChunk`** objects.

These chunks can be combined to create the full message.

In [ ]:
chunks = []

# Example:
# for chunk in model.stream("Hi"):
#     chunks.append(chunk)
#     print(chunk.text)

print("Streaming gives the response in smaller chunks.")

## 23. Token Usage

Models process text using **tokens**.

An `AIMessage` can contain usage information in `usage_metadata`.

It may include:

- Input tokens
- Output tokens
- Total tokens
- Other provider-specific token details

This can help you understand how much model usage occurred.

In [ ]:
# Example:
# response = model.invoke("Hello!")
# print(response.usage_metadata)

print("usage_metadata can contain token counts.")

## 24. Serialization

**Serialization** means converting a message object into a normal Python data structure so it can be stored.

You can later convert it back into a message object.

This is useful for:
- Saving conversation history
- Loading old conversations
- Resuming a session

In [ ]:
from langchain.messages import HumanMessage
from langchain_core.load import dumpd, load

message = HumanMessage("What is the capital of France?")

serialized = dumpd(message)

print(serialized)

# Restore the message
# restored = load(serialized)
# print(restored)

print("Messages can be serialized for storage.")

## 25. Important Security Warning

Be careful with `load()`.

The documentation warns that `load()` creates Python objects and can cause side effects during deserialization.

**Never call `load()` on data from an untrusted or unauthenticated source.**

## 26. Messages with Chat Models

Chat models accept a sequence of messages and return an `AIMessage`.

A simple conversation can keep a growing list of messages:

**User message → AI response → User message → AI response**

This allows the application to keep conversation history.

For long conversations, LangChain also provides methods for managing the history, such as trimming or summarizing messages.

## 27. Quick Revision

| Topic | Simple meaning |
|---|---|
| Message | Information sent to or from a model |
| Role | Tells who sent the message |
| Content | Actual information |
| Metadata | Extra information |
| SystemMessage | Instructions for the AI |
| HumanMessage | User input |
| AIMessage | AI response |
| ToolMessage | Result from a tool |
| Tool call | AI asks a tool/function to do something |
| Content block | Standard format for content |
| Multimodal | Text + image/audio/video/file |
| AIMessageChunk | Small piece of a streamed response |
| Token usage | Information about model token use |
| Serialization | Convert message to storable data |


## 28. Final Summary

The easiest way to remember LangChain Messages is:

**Message = Role + Content + Optional Metadata**

### Main message types
- `SystemMessage` → tells the AI what to do
- `HumanMessage` → user input
- `AIMessage` → AI output
- `ToolMessage` → tool result

### Main idea

Use a simple **string** for a simple one-time question.

Use **messages** when you need:
- Conversation history
- System instructions
- Tools
- Images
- Audio
- Video
- Files

LangChain's standard message and content-block system makes these features easier to use across different model providers.